# Data merge and feature engineering


In [1]:
%pip install pyarrow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: C:\Users\Uw11\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import string

import re
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm.notebook import tqdm
from tqdm import tqdm
import pymorphy3
from collections import Counter

import os
from pathlib import Path
import json

In [3]:
df_weather = pd.read_csv("../data/weather_processed.csv")
df_war_events = pd.read_csv("../data/war_events_processed.csv")
df_isw_matrix = pd.read_csv("../data/isw_processed_svd.csv")

In [4]:
pd.set_option('display.max_columns', None)

In [5]:
df_weather['datetime_hour'] = pd.to_datetime(df_weather['datetime_hour'], errors="coerce")
df_war_events['datetime_hour'] = pd.to_datetime(df_war_events['datetime_hour'], errors="coerce")

In [6]:
df_final = pd.merge(
    df_weather, 
    df_war_events[['datetime_hour', 'region_id', 'alarm_active', 'alarm_minutes_in_hour']], 
    on=['datetime_hour', 'region_id'], 
    how='left'
)
df_final.sample(20)

,datetime_hour,region_id,city_latitude,city_longitude,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,day_snow,day_snowdepth,day_windgust,day_windspeed,day_winddir,day_pressure,day_cloudcover,day_solarradiation,day_solarenergy,day_uvindex,day_moonphase,hour_temp,hour_feelslike,hour_humidity,hour_precip,hour_precipprob,hour_snow,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,day_conditions_simple_Clear,day_conditions_simple_Cloudy,day_conditions_simple_Rain,day_conditions_simple_Snow,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,year,month,day_of_week,hour,city_name,city_timezone,day_datetime,day_sunrise,day_sunset,hour_datetime,region_key,alarm_active,alarm_minutes_in_hour
261180,2023-05-23 12:00:00,15,46.47250,30.73710,18.7,13.6,73.5,1.2,100.0,12.50,0.0,0.0,29.2,14.8,8.2,1012.1,45.8,242.2,21.1,8.0,0.12,21.1,21.1,65.56,0.0,0.0,0.0,26.3,3.6,100.0,1012.1,90.0,738.7,2.7,7.0,False,False,True,False,False,True,False,False,2023,5,1,12,Odesa,Europe/Kiev,2023-05-23,05:15:26,20:32:53,12:00:00,Одеська,0,0.000000
558772,2024-10-21 05:00:00,6,50.25360,28.66540,5.1,0.2,74.8,0.0,0.0,0.00,0.0,0.0,11.5,7.2,215.9,1034.0,22.3,0.0,0.0,0.0,0.63,-0.7,-0.7,99.27,0.0,0.0,0.0,5.0,3.2,263.7,1035.0,53.2,5.6,0.1,0.0,False,True,False,False,False,True,False,False,2024,10,0,5,Zhytomyr,Europe/Kiev,2024-10-21,07:37:51,18:01:13,05:00:00,Житомирська,1,13.633333
195306,2023-01-29 02:00:00,21,46.64010,32.61420,-1.5,-2.8,90.4,0.5,100.0,4.17,1.8,3.0,41.0,25.2,5.8,1014.8,98.5,26.0,2.2,1.0,0.25,-1.4,-6.5,97.10,0.0,0.0,0.2,29.2,16.9,12.5,1013.0,99.3,0.0,0.0,0.0,False,False,False,True,False,True,False,False,2023,1,6,2,Kherson,Europe/Kiev,2023-01-29,07:17:44,16:48:05,02:00:00,Херсонська,0,0.000000
499634,2024-07-10 13:00:00,4,48.47530,35.01600,29.4,14.0,42.1,0.0,0.0,0.00,0.0,0.0,32.4,18.0,63.4,1016.7,44.0,308.6,26.5,9.0,0.14,35.2,33.5,22.16,0.0,0.0,0.0,29.5,11.9,68.6,1017.0,40.1,852.1,3.1,9.0,False,True,False,False,False,True,False,False,2024,7,2,13,Dnipro,Europe/Kiev,2024-07-10,04:49:36,20:40:47,13:00:00,Дніпропетровська,0,0.000000
49441,2022-05-20 21:00:00,3,50.74690,25.32630,16.4,3.2,45.4,0.0,0.0,0.00,0.0,0.0,28.1,26.5,232.1,1021.0,64.4,255.8,22.3,7.0,0.64,22.1,22.1,40.19,0.0,0.0,0.0,11.2,13.6,256.0,1013.6,47.7,35.0,0.1,0.0,False,True,False,False,False,True,False,False,2022,5,4,21,Lutsk,Europe/Kiev,2022-05-20,05:23:28,21:07:56,21:00:00,Волинська,0,0.000000
232213,2023-04-03 05:00:00,16,49.58790,34.55170,7.4,5.8,89.4,8.0,100.0,8.33,0.0,0.0,32.0,19.4,71.5,1007.6,99.3,33.6,3.0,1.0,0.41,6.9,3.6,85.88,0.0,0.0,0.0,29.5,18.4,85.6,1006.0,100.0,0.0,0.0,0.0,False,False,True,False,False,True,False,False,2023,4,0,5,Poltava,Europe/Kiev,2023-04-03,06:15:33,19:15:49,05:00:00,Полтавська,0,0.000000
295691,2023-07-22 10:00:00,14,46.97340,31.98520,23.3,15.4,64.7,0.6,100.0,4.17,0.0,0.0,42.5,21.6,169.0,1014.0,42.2,212.3,18.3,7.0,0.15,24.7,24.7,52.03,0.0,0.0,0.0,32.8,16.6,149.0,1014.0,44.5,488.8,1.8,5.0,False,False,True,False,False,True,False,False,2023,7,5,10,Mykolaiv,Europe/Kiev,2023-07-22,05:19:14,20:37:14,10:00:00,Миколаївська,0,0.000000
59198,2022-06-06 19:00:00,17,50.61927,26.25131,18.3,11.5,68.9,0.0,0.0,0.00,0.0,0.0,15.5,8.3,101.8,1019.1,58.6,294.0,25.4,9.0,0.22,23.0,23.0,45.50,0.0,0.0,0.0,15.1,7.9,121.6,1017.0,59.6,249.0,0.9,2.0,False,True,False,False,False,True,False,False,2022,6,0,19,Rivne,Europe/Kiev,2022-06-06,05:04:55,21:22:59,19:00:00,Рівненська,0,0.000000
323784,2023-09-09 05:00:00,2,49.23360,28.44860,16.4,9.4,66.4,0.0,0.0,0.00,0.0,0.0,35.3,14.4,4.4,1020.8,42.3,202.2,17.4,7.0,0.81,10.8,10.8,84.54,0.0,0.0,0.0,13.3,9.4,323.2,1019.0,45.5,0.0,0.0,0.0,False,True,False,False,False,True,False,False,2023,9,5,5,Vinnytsia,Europe/Kiev,2023-09-09,06:33:12,19:33:16,05:00:00,Вінницька,0,0.000000
633098,2025-02-27 06:00:00,4,48.47530,35.01600,-4.1,-9.0,70.8,0.0,0.0,0.00,0.0,0.5,19.8,7.2,143.8,1029.6,3.8,151.4,13.1,5.0,0.98,

In [7]:
df_final = df_final.sort_values(['region_id', 'datetime_hour'])
df_final.head()

,datetime_hour,region_id,city_latitude,city_longitude,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,day_snow,day_snowdepth,day_windgust,day_windspeed,day_winddir,day_pressure,day_cloudcover,day_solarradiation,day_solarenergy,day_uvindex,day_moonphase,hour_temp,hour_feelslike,hour_humidity,hour_precip,hour_precipprob,hour_snow,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,day_conditions_simple_Clear,day_conditions_simple_Cloudy,day_conditions_simple_Rain,day_conditions_simple_Snow,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,year,month,day_of_week,hour,city_name,city_timezone,day_datetime,day_sunrise,day_sunset,hour_datetime,region_key,alarm_active,alarm_minutes_in_hour
0,2022-02-24 00:00:00,2,49.2336,28.4486,2.8,-0.3,80.5,0.3,100.0,4.17,0.0,0.0,27.7,10.8,312.0,1023.4,82.5,60.4,5.4,3.0,0.77,2.1,-0.5,91.76,0.0,0.0,0.0,16.6,8.6,279.6,1021.0,91.1,0.0,0.1,0.0,False,False,False,True,False,True,False,False,2022,2,3,0,Vinnytsia,Europe/Kiev,2022-02-24,06:58:49,17:40:52,00:00:00,Вінницька,0,0.0
24,2022-02-24 01:00:00,2,49.2336,28.4486,2.8,-0.3,80.5,0.3,100.0,4.17,0.0,0.0,27.7,10.8,312.0,1023.4,82.5,60.4,5.4,3.0,0.77,2.1,-0.4,89.80,0.0,0.0,0.0,16.6,8.3,289.2,1021.0,97.9,0.0,0.1,0.0,False,False,False,True,False,True,False,False,2022,2,3,1,Vinnytsia,Europe/Kiev,2022-02-24,06:58:49,17:40:52,01:00:00,Вінницька,0,0.0
48,2022-02-24 02:00:00,2,49.2336,28.4486,2.8,-0.3,80.5,0.3,100.0,4.17,0.0,0.0,27.7,10.8,312.0,1023.4,82.5,60.4,5.4,3.0,0.77,2.0,-0.3,90.44,0.0,0.0,0.0,27.7,7.6,287.6,1021.0,98.2,0.0,0.1,0.0,False,False,False,True,False,True,False,False,2022,2,3,2,Vinnytsia,Europe/Kiev,2022-02-24,06:58:49,17:40:52,02:00:00,Вінницька,0,0.0
72,2022-02-24 03:00:00,2,49.2336,28.4486,2.8,-0.3,80.5,0.3,100.0,4.17,0.0,0.0,27.7,10.8,312.0,1023.4,82.5,60.4,5.4,3.0,0.77,1.9,-0.3,91.75,0.0,0.0,0.0,15.1,7.2,288.9,1021.0,98.8,0.0,0.1,0.0,False,False,False,True,False,True,False,False,2022,2,3,3,Vinnytsia,Europe/Kiev,2022-02-24,06:58:49,17:40:52,03:00:00,Вінницька,0,0.0
96,2022-02-24 04:00:00,2,49.2336,28.4486,2.8,-0.3,80.5,0.3,100.0,4.17,0.0,0.0,27.7,10.8,312.0,1023.4,82.5,60.4,5.4,3.0,0.77,1.8,-0.2,92.40,0.0,0.0,0.0,13.7,6.5,290.4,1021.0,100.0,0.0,0.1,0.0,False,False,False,True,False,True,False,False,2022,2,3,4,Vinnytsia,Europe/Kiev,2022-02-24,06:58:49,17:40:52,04:00:00,Вінницька,0,0.0


In [8]:
df_final.info()

<class 'pandas.DataFrame'>
Index: 634680 entries, 0 to 634679
Data columns (total 56 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   datetime_hour                  634680 non-null  datetime64[us]
 1   region_id                      634680 non-null  int64         
 2   city_latitude                  634680 non-null  float64       
 3   city_longitude                 634680 non-null  float64       
 4   day_temp                       634680 non-null  float64       
 5   day_dew                        634680 non-null  float64       
 6   day_humidity                   634680 non-null  float64       
 7   day_precip                     634680 non-null  float64       
 8   day_precipprob                 634680 non-null  float64       
 9   day_precipcover                634680 non-null  float64       
 10  day_snow                       634680 non-null  float64       
 11  day_snowdepth   

In [9]:
df_final = df_final.sort_values(["region_id", "datetime_hour"])

df_final["alarm_lag_1"] = df_final.groupby("region_id")["alarm_active"].shift(1)
df_final["alarm_lag_3"] = df_final.groupby("region_id")["alarm_active"].shift(3)
df_final["alarm_lag_6"] = df_final.groupby("region_id")["alarm_active"].shift(6)
df_final["alarm_lag_12"] = df_final.groupby("region_id")["alarm_active"].shift(12)
#df_final['alarm_minutes_lag1'] = (df_final.groupby('region_id')['alarm_minutes_in_hour'].shift(1).fillna(0))
#df_final['alarm_minutes_lag3'] = (df_final.groupby('region_id')['alarm_minutes_in_hour'].shift(3).fillna(0))

lag_cols = ["alarm_lag_1","alarm_lag_3","alarm_lag_6","alarm_lag_12"]
df_final[lag_cols] = df_final[lag_cols].fillna(0)

df_final['alarms_in_last_24h'] = (df_final.groupby('region_id')['alarm_active'].transform(lambda x: x.shift(1).rolling(24, min_periods=1).sum()))

df_final['is_weekend'] = df_final['day_of_week'].isin([5, 6]).astype(int)
df_final['is_night'] = ((df_final['hour'] >= 23) | (df_final['hour'] <= 6)).astype(int)

hourly_total = df_final.groupby('datetime_hour')['alarm_active'].sum().shift(1)
df_final['total_active_alarms_lag1'] = df_final['datetime_hour'].map(hourly_total)

df_final.sample(10)

,datetime_hour,region_id,city_latitude,city_longitude,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,day_snow,day_snowdepth,day_windgust,day_windspeed,day_winddir,day_pressure,day_cloudcover,day_solarradiation,day_solarenergy,day_uvindex,day_moonphase,hour_temp,hour_feelslike,hour_humidity,hour_precip,hour_precipprob,hour_snow,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,day_conditions_simple_Clear,day_conditions_simple_Cloudy,day_conditions_simple_Rain,day_conditions_simple_Snow,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,year,month,day_of_week,hour,city_name,city_timezone,day_datetime,day_sunrise,day_sunset,hour_datetime,region_key,alarm_active,alarm_minutes_in_hour,alarm_lag_1,alarm_lag_3,alarm_lag_6,alarm_lag_12,alarms_in_last_24h,is_weekend,is_night,total_active_alarms_lag1
5106,2022-03-04 20:00:00,21,46.64010,32.61420,2.7,0.0,83.5,0.1,100.0,4.17,0.0,0.0,37.8,24.8,293.4,1011.1,80.4,92.4,6.6,3.0,0.04,2.0,-1.7,89.14,0.0,0.0,0.0,30.6,13.3,267.8,1012.0,78.9,5.6,0.1,0.0,False,False,False,True,False,True,False,False,2022,3,4,20,Kherson,Europe/Kiev,2022-03-04,06:23:55,17:39:26,20:00:00,Херсонська,0,0.000000,0.0,0.0,0.0,0.0,0.0,0,0,3.0
359600,2023-11-10 09:00:00,10,50.45060,30.52430,6.2,3.1,82.2,0.0,0.0,0.00,0.0,0.0,42.5,21.2,169.2,1018.7,62.8,69.7,5.9,3.0,0.90,1.7,-2.6,94.42,0.0,0.0,0.0,28.8,16.2,162.9,1021.0,97.8,118.2,0.4,1.0,False,True,False,False,False,True,False,False,2023,11,4,9,Kyiv,Europe/Kiev,2023-11-10,07:03:11,16:19:46,09:00:00,Київська,0,0.000000,0.0,0.0,1.0,0.0,3.0,0,0,0.0
607778,2025-01-14 07:00:00,4,48.47530,35.01600,-1.6,-4.9,78.7,0.0,0.0,0.00,0.0,0.0,30.6,18.0,271.4,1031.8,82.1,32.6,2.9,2.0,0.50,-2.5,-6.3,81.69,0.0,0.0,0.0,19.8,9.7,288.1,1033.0,93.7,0.0,0.0,0.0,False,True,False,False,False,True,False,False,2025,1,1,7,Dnipro,Europe/Kiev,2025-01-14,07:27:22,16:11:12,07:00:00,Дніпропетровська,0,0.000000,1.0,1.0,1.0,1.0,13.0,0,0,4.0
505928,2024-07-21 11:00:00,10,50.45060,30.52430,23.3,14.9,62.6,0.0,0.0,0.00,0.0,0.0,16.6,6.5,193.9,1010.4,86.5,214.0,18.6,6.0,0.50,27.9,27.7,42.24,0.0,0.0,0.0,16.2,5.0,213.1,1011.0,96.0,618.9,2.2,6.0,False,True,False,False,False,True,False,False,2024,7,6,11,Kyiv,Europe/Kiev,2024-07-21,05:11:06,20:56:56,11:00:00,Київська,0,0.000000,0.0,0.0,0.0,0.0,4.0,1,0,3.0
202426,2023-02-10 11:00:00,13,49.84440,24.02540,-4.3,-9.0,70.6,0.0,0.0,0.00,0.0,9.4,34.6,21.6,251.5,1035.7,63.0,103.8,8.9,4.0,0.66,-3.1,-8.2,62.27,0.0,0.0,0.0,32.8,14.4,260.0,1036.0,30.0,312.3,1.1,3.0,False,True,False,False,False,True,False,False,2023,2,4,11,Lviv,Europe/Kiev,2023-02-10,07:43:35,17:33:25,11:00:00,Львівська,1,58.983333,1.0,1.0,0.0,0.0,3.0,0,0,24.0
616408,2025-01-29 06:00:00,19,49.55270,25.58890,7.1,3.4,79.3,0.4,100.0,4.17,0.0,0.0,41.0,18.4,197.8,1012.6,48.7,61.1,5.3,3.0,0.00,4.0,0.7,94.52,0.0,0.0,0.0,19.4,13.7,171.1,1009.0,98.1,0.0,0.0,0.0,False,False,True,False,False,True,False,False,2025,1,2,6,Ternopil,Europe/Kiev,2025-01-29,07:53:51,17:08:26,06:00:00,Тернопільська,0,0.000000,0.0,0.0,0.0,0.0,1.0,0,1,8.0
375805,2023-12-08 12:00:00,16,49.58790,34.55170,-2.4,-3.7,91.3,7.0,100.0,8.33,5.8,4.6,42.5,24.5,95.0,1023.9,100.0,10.2,0.9,0.0,0.85,-2.4,-8.1,93.53,0.0,0.0,0.2,36.4,18.7,85.7,1024.0,100.0,48.9,0.2,0.0,False,False,False,True,False,True,False,False,2023,12,4,12,Poltava,Europe/Kiev,2023-12-08,07:24:51,15:42:05,12:00:00,Полтавська,0,0.000000,0.0,0.0,1.0,0.0,11.0,0,0,1.0
171616,2022-12-18 23:00:00,19,49.55270,25.58890,-5.9,-7.9,86.1,0.2,100.0,8.33,0.0,5.5,45.0,27.4,306.6,1034.5,97.1,55.0,4.9,3.0,0.81,-8.1,-13.2,88.24,0.0,0.0,0.0,18.4,10.4,326.9,1042.0,100.0,0.0,0.1,0.0,False,False,False,True,False,True,False,False,2022,12,6,23,Ternopil,Europe/Kiev,2022-12-18,08:09:43,16:18:41,23:00:00,Тернопільська,0,0.000000,0.0,0.0,0.0,0.0,0.0,1,1,0.0
30101,2022-04-17 07:00:00,7,48.62636,22.28514,5.0,-4.0,53.5,0.0,0.0,0.00,0.0,2.5,47.2,22.5,14.0,1024.5,34.8,262.5,22.6

In [10]:
neighbouring_regions = {
    1: [21],
    2: [6, 10, 11, 15, 22, 23, 24],
    3: [13, 17],
    4: [5, 8, 11, 14, 16, 20, 21],
    5: [4, 8, 12, 20],
    6: [2, 10, 17, 22],
    7: [9, 13],
    8: [4, 5, 21],
    9: [7, 13, 19, 24],
    10: [2, 6, 16, 23, 25],
    11: [2, 4, 14, 15, 16, 23],
    12: [5, 20],
    13: [3, 7, 9, 17, 19],
    14: [4, 11, 15, 21],
    15: [2, 11, 14],
    16: [4, 10, 11, 18, 20, 23, 25],
    17: [3, 6, 13, 19, 22],
    18: [16, 20, 25],
    19: [9, 13, 17, 22, 24],
    20: [4, 5, 12, 16, 18],
    21: [1, 4, 8, 14],
    22: [2, 6, 17, 19, 24],
    23: [2, 10, 11, 16],
    24: [2, 9, 19, 22],
    25: [10, 16, 18], 
    26: [10]
}

alarms_matrix = df_final.pivot_table(
    index='datetime_hour',
    columns='region_id',
    values='alarm_active',
    fill_value=0
)

neighbour_alarm_matrix = pd.DataFrame(index=alarms_matrix.index)

for region, neighbours in neighbouring_regions.items():
    valid_neighbours = [n for n in neighbours if n in alarms_matrix.columns]
    
    if valid_neighbours:
        neighbour_alarm_matrix[region] = alarms_matrix[valid_neighbours].sum(axis=1)
    else:
        neighbour_alarm_matrix[region] = 0

neighbour_alarm_matrix = neighbour_alarm_matrix.shift(1)

neighbour_alarm_long = (neighbour_alarm_matrix.stack().reset_index())

neighbour_alarm_long.columns = ['datetime_hour','region_id','neighbour_alarms']

df_final = df_final.merge(neighbour_alarm_long,on=['datetime_hour', 'region_id'], how='left')

In [11]:
def hours_since_last_alarm(series):
    shifted = series.shift(1).fillna(0)
    result = []
    count = 0
    for val in shifted:
        if val == 1:
            count = 0
        else:
            count += 1
        result.append(count)
    return pd.Series(result, index=series.index)

df_final['hours_since_last_alarm'] = (df_final.groupby('region_id')['alarm_active'].transform(hours_since_last_alarm))

In [12]:
df_final.sample(20)

,datetime_hour,region_id,city_latitude,city_longitude,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,day_snow,day_snowdepth,day_windgust,day_windspeed,day_winddir,day_pressure,day_cloudcover,day_solarradiation,day_solarenergy,day_uvindex,day_moonphase,hour_temp,hour_feelslike,hour_humidity,hour_precip,hour_precipprob,hour_snow,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,day_conditions_simple_Clear,day_conditions_simple_Cloudy,day_conditions_simple_Rain,day_conditions_simple_Snow,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,year,month,day_of_week,hour,city_name,city_timezone,day_datetime,day_sunrise,day_sunset,hour_datetime,region_key,alarm_active,alarm_minutes_in_hour,alarm_lag_1,alarm_lag_3,alarm_lag_6,alarm_lag_12,alarms_in_last_24h,is_weekend,is_night,total_active_alarms_lag1,neighbour_alarms,hours_since_last_alarm
155353,2024-10-14 19:00:00,7,48.6264,22.2851,7.5,3.0,75.5,0.000,0.0,0.00,0.0,0.0,24.8,9.0,239.3,1018.8,39.4,92.9,8.1,4.0,0.39,8.9,8.0,63.78,0.0,0.0,0.0,12.2,6.8,211.0,1019.6,36.0,15.1,0.1,0.0,False,True,False,False,False,True,False,False,2024,10,0,19,Uzhgorod,Europe/Uzhgorod,2024-10-14,07:49:53,18:42:56,19:00:00,Закарпатська,0,0.000000,0.0,0.0,0.0,0.0,0.0,0,0,0.0,0.0,67
227148,2023-12-05 14:00:00,10,50.4506,30.5243,-4.1,-6.1,86.2,0.100,100.0,4.17,0.1,7.2,25.9,14.0,172.2,1025.4,93.0,36.5,3.1,2.0,0.75,-2.3,-5.2,76.90,0.0,0.0,0.0,17.6,7.2,130.0,1025.5,100.0,121.2,0.4,1.0,False,False,False,True,False,True,False,False,2023,12,1,14,Kyiv,Europe/Kiev,2023-12-05,07:41:17,15:55:16,14:00:00,Київська,0,0.000000,0.0,0.0,0.0,0.0,0.0,0,0,3.0,0.0,38
278430,2023-09-29 14:00:00,13,49.8444,24.0254,16.1,10.1,72.0,0.000,0.0,0.00,0.0,0.0,24.8,11.5,167.6,1023.2,3.3,168.5,14.6,6.0,0.50,24.9,24.9,38.48,0.0,0.0,0.0,24.5,11.2,172.7,1023.0,0.0,585.7,2.1,6.0,True,False,False,False,True,False,False,False,2023,9,4,14,Lviv,Europe/Kiev,2023-09-29,07:20:06,19:07:46,14:00:00,Львівська,0,0.000000,0.0,0.0,0.0,0.0,0.0,0,0,4.0,0.0,66
383867,2023-09-15 07:00:00,17,50.6193,26.2513,15.9,12.5,81.5,0.790,100.0,4.17,0.0,0.0,32.8,16.6,354.2,1021.4,84.2,66.3,5.7,2.0,0.00,14.9,14.9,93.13,0.0,0.0,0.0,26.6,13.0,333.6,1020.0,100.0,0.1,0.0,0.0,False,False,True,False,False,True,False,False,2023,9,4,7,Rivne,Europe/Kiev,2023-09-15,06:49:45,19:30:02,07:00:00,Рівненська,1,35.833333,1.0,0.0,0.0,0.0,1.0,0,0,8.0,3.0,0
559846,2022-08-30 14:00:00,24,48.2924,25.9355,20.0,17.5,86.2,0.351,100.0,4.17,0.0,0.0,19.9,10.8,317.4,1018.6,89.3,147.8,12.8,5.0,0.09,21.9,21.9,74.57,0.0,0.0,0.0,16.4,9.4,321.0,1019.4,95.8,253.0,0.9,3.0,False,False,True,False,False,True,False,False,2022,8,1,14,Chernivtsi,Europe/Kiev,2022-08-30,06:30:43,20:02:15,14:00:00,Чернівецька,0,0.000000,0.0,0.0,0.0,0.0,0.0,0,0,1.0,0.0,42
552445,2024-11-01 04:00:00,23,49.4407,32.0637,10.4,6.0,74.4,0.000,0.0,0.00,0.0,0.0,50.8,25.9,262.6,1012.9,76.3,31.2,2.7,1.0,0.00,8.4,5.6,79.72,0.0,0.0,0.0,30.2,17.6,247.3,1016.0,100.0,0.0,0.0,0.0,False,True,False,False,False,True,False,False,2024,11,4,4,Cherkasy,Europe/Kiev,2024-11-01,06:40:33,16:29:26,04:00:00,Черкаська,1,17.366667,0.0,1.0,0.0,0.0,7.0,0,1,8.0,2.0,2
493387,2024-02-18 03:00:00,21,46.6401,32.6142,0.8,0.1,95.1,0.100,100.0,4.17,0.0,0.0,30.2,15.5,288.4,1027.6,92.5,48.4,4.3,2.0,0.29,-0.7,-0.7,91.58,0.0,0.0,0.0,2.5,2.5,251.8,1028.0,94.6,0.0,0.0,0.0,False,False,True,False,False,True,False,False,2024,2,6,3,Kherson,Europe/Kiev,2024-02-18,06:49:40,17:18:07,03:00:00,Херсонська,0,0.000000,0.0,0.0,0.0,1.0,4.0,1,1,5.0,2.0,6
179095,2024-06-24 04:00:00,8,47.8289,35.1626,23.5,16.1,64.6,0.000,0.0,0.00,0.0,0.0,34.2,22.3,320.1,1010.0,49.7,296.9,25.6,8.0,0.58,21.8,21.8,81.53,0.0,0.0,0.0,31.7,18.4,333.7,1010.0,12.9,0.0,0.0,0.0,False,True,False,False,True,False,False,False,2024,6,0,4,Zaporozhye,Europe/Zaporozhye,2024-06-24,04:41:35,20:42:13,04:00:00,Запорізька,0,0.000000,0.0,0.0,1.0,0.0,7.0,0,1,7.0,1.0,1
394552,2024-12-03 

In [13]:
df_isw_matrix.sample(10)

,date,isw_topic_0,isw_topic_1,isw_topic_2,isw_topic_3,isw_topic_4,isw_topic_5,isw_topic_6,isw_topic_7,isw_topic_8,isw_topic_9,isw_topic_10,isw_topic_11,isw_topic_12,isw_topic_13,isw_topic_14,isw_topic_15,isw_topic_16,isw_topic_17,isw_topic_18,isw_topic_19,isw_topic_20,isw_topic_21,isw_topic_22,isw_topic_23,isw_topic_24,isw_topic_25,isw_topic_26,isw_topic_27,isw_topic_28,isw_topic_29,isw_topic_30,isw_topic_31,isw_topic_32,isw_topic_33,isw_topic_34,isw_topic_35,isw_topic_36,isw_topic_37,isw_topic_38,isw_topic_39,isw_topic_40,isw_topic_41,isw_topic_42,isw_topic_43,isw_topic_44,isw_topic_45,isw_topic_46,isw_topic_47,isw_topic_48,isw_topic_49,isw_topic_50,isw_topic_51,isw_topic_52,isw_topic_53,isw_topic_54,isw_topic_55,isw_topic_56,isw_topic_57,isw_topic_58,isw_topic_59,isw_topic_60,isw_topic_61,isw_topic_62,isw_topic_63,isw_topic_64,isw_topic_65,isw_topic_66,isw_topic_67,isw_topic_68,isw_topic_69,isw_topic_70,isw_topic_71,isw_topic_72,isw_topic_73,isw_topic_74,isw_topic_75,isw_topic_76,isw_topic_77,isw_topic_78,isw_topic_79,isw_topic_80,isw_topic_81,isw_topic_82,isw_topic_83,isw_topic_84,isw_topic_85,isw_topic_86,isw_topic_87,isw_topic_88,isw_topic_89,isw_topic_90,isw_topic_91,isw_topic_92,isw_topic_93,isw_topic_94,isw_topic_95,isw_topic_96,isw_topic_97,isw_topic_98,isw_topic_99,isw_topic_100,isw_topic_101,isw_topic_102,isw_topic_103,isw_topic_104,isw_topic_105,isw_topic_106,isw_topic_107,isw_topic_108,isw_topic_109,isw_topic_110,isw_topic_111,isw_topic_112,isw_topic_113,isw_topic_114,isw_topic_115,isw_topic_116,isw_topic_117,isw_topic_118,isw_topic_119,isw_topic_120,isw_topic_121,isw_topic_122,isw_topic_123,isw_topic_124,isw_topic_125,isw_topic_126,isw_topic_127,isw_topic_128,isw_topic_129,isw_topic_130,isw_topic_131,isw_topic_132,isw_topic_133,isw_topic_134,isw_topic_135,isw_topic_136,isw_topic_137,isw_topic_138,isw_topic_139,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149
136,2022-07-09,0.744766,-0.206271,0.208248,-0.001226,-0.063301,0.104104,0.071302,-0.101936,-0.002449,0.012457,-0.116096,-0.127555,0.052244,-0.033801,-0.017166,0.046522,-0.055906,0.009067,-0.068595,-0.031481,0.006475,-0.082863,-0.033245,-0.025740,0.054279,0.025302,-0.026230,-0.044486,0.005255,-0.001249,0.072920,0.013327,-0.009163,0.016965,0.018714,0.011645,0.025238,-0.023150,0.002279,0.042971,-0.021873,-0.013869,-0.056613,-0.041758,0.000967,0.042961,0.040352,-0.068175,-0.043832,-0.012052,-0.008501,0.005472,-0.029390,0.002899,0.057730,0.031018,0.011728,-0.030052,-0.006495,0.030576,-0.022661,-0.036235,0.063426,0.026370,0.025170,0.025443,-0.030314,0.004432,-0.028582,-0.042687,0.039736,0.027267,-0.005573,-0.014119,-0.029532,0.009991,-0.029268,0.013783,-0.043374,-0.006110,-0.027900,0.002715,0.007302,0.015827,0.001589,-0.018015,-0.001485,-0.053690,-0.007329,-0.047021,-0.005765,0.006623,0.007871,-0.007514,0.002737,0.005117,-0.016600,0.003849,-0.018555,-0.007650,0.011154,-0.003941,0.019259,0.008886,-0.045556,-0.017753,0.004719,-0.004430,0.013214,0.018799,0.009835,-0.034650,-0.002944,-0.010044,-0.004507,-0.058413,-0.008916,-0.002448,0.017348,-0.007322,-0.001819,0.048098,0.039007,0.017339,-0.045625,0.003107,-0.012315,0.051055,0.037209,0.028696,0.013277,-0.036274,0.013106,0.005894,-0.011250,0.010466,0.055308,-0.011417,-0.002946,-0.015453,-0.019784,0.023989,0.065556,-0.008810,0.020015,-0.014550,0.025466,-0.020187,-0.000549,0.014066
1327,2025-10-22,0.753436,0.270383,0.030635,-0.184268,-0.014613,-0.170559,-0.237476,-0.021032,-0.221238,-0.071660,-0.093766,-0.032702,-0.064511,-0.099170,-0.128805,0.022666,0.022785,-0.009746,0.021250,-0.028994,0.068013,0.000166,0.027887,-0.039335,0.021479,-0.052618,0.026810,0.002537,-0.045076,-0.055689,0.027172,0.009938,0.025040,0.006673,0.001364,-0.012282,-0.023637,-0.023138,0.023443,0.037326,-0.009204,-0.025638,0.013522,-0.021824,0.028498,0.007547,0.000842,-0.065556,-0.006029,0.009895,-0.015914,0.019658,0.013272,0.055888,-0.027435,0.000605,0.000214,0.03

In [14]:
#df_isw_matrix["date"] = pd.to_datetime(df_isw_matrix["date"]) + pd.Timedelta(days=1)    ТУТ ЗСУВ ІСВ НА + 1 ДЕНЬ
df_isw_matrix = df_isw_matrix.rename(columns={'date': 'day_datetime'})

In [15]:
df_final['day_datetime'] = pd.to_datetime(df_final['day_datetime']).dt.date
df_isw_matrix['day_datetime'] = pd.to_datetime(df_isw_matrix['day_datetime']).dt.date
df_isw_matrix.fillna(0, inplace=True)
df_isw_matrix.isna().sum()

day_datetime     0
isw_topic_0      0
isw_topic_1      0
isw_topic_2      0
isw_topic_3      0
                ..
isw_topic_145    0
isw_topic_146    0
isw_topic_147    0
isw_topic_148    0
isw_topic_149    0
Length: 151, dtype: int64

In [16]:
df_final = df_final.merge(df_isw_matrix, on="day_datetime", how="left")

In [17]:
df_final.head(10)

,datetime_hour,region_id,city_latitude,city_longitude,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,day_snow,day_snowdepth,day_windgust,day_windspeed,day_winddir,day_pressure,day_cloudcover,day_solarradiation,day_solarenergy,day_uvindex,day_moonphase,hour_temp,hour_feelslike,hour_humidity,hour_precip,hour_precipprob,hour_snow,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,day_conditions_simple_Clear,day_conditions_simple_Cloudy,day_conditions_simple_Rain,day_conditions_simple_Snow,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,year,month,day_of_week,hour,city_name,city_timezone,day_datetime,day_sunrise,day_sunset,hour_datetime,region_key,alarm_active,alarm_minutes_in_hour,alarm_lag_1,alarm_lag_3,alarm_lag_6,alarm_lag_12,alarms_in_last_24h,is_weekend,is_night,total_active_alarms_lag1,neighbour_alarms,hours_since_last_alarm,isw_topic_0,isw_topic_1,isw_topic_2,isw_topic_3,isw_topic_4,isw_topic_5,isw_topic_6,isw_topic_7,isw_topic_8,isw_topic_9,isw_topic_10,isw_topic_11,isw_topic_12,isw_topic_13,isw_topic_14,isw_topic_15,isw_topic_16,isw_topic_17,isw_topic_18,isw_topic_19,isw_topic_20,isw_topic_21,isw_topic_22,isw_topic_23,isw_topic_24,isw_topic_25,isw_topic_26,isw_topic_27,isw_topic_28,isw_topic_29,isw_topic_30,isw_topic_31,isw_topic_32,isw_topic_33,isw_topic_34,isw_topic_35,isw_topic_36,isw_topic_37,isw_topic_38,isw_topic_39,isw_topic_40,isw_topic_41,isw_topic_42,isw_topic_43,isw_topic_44,isw_topic_45,isw_topic_46,isw_topic_47,isw_topic_48,isw_topic_49,isw_topic_50,isw_topic_51,isw_topic_52,isw_topic_53,isw_topic_54,isw_topic_55,isw_topic_56,isw_topic_57,isw_topic_58,isw_topic_59,isw_topic_60,isw_topic_61,isw_topic_62,isw_topic_63,isw_topic_64,isw_topic_65,isw_topic_66,isw_topic_67,isw_topic_68,isw_topic_69,isw_topic_70,isw_topic_71,isw_topic_72,isw_topic_73,isw_topic_74,isw_topic_75,isw_topic_76,isw_topic_77,isw_topic_78,isw_topic_79,isw_topic_80,isw_topic_81,isw_topic_82,isw_topic_83,isw_topic_84,isw_topic_85,isw_topic_86,isw_topic_87,isw_topic_88,isw_topic_89,isw_topic_90,isw_topic_91,isw_topic_92,isw_topic_93,isw_topic_94,isw_topic_95,isw_topic_96,isw_topic_97,isw_topic_98,isw_topic_99,isw_topic_100,isw_topic_101,isw_topic_102,isw_topic_103,isw_topic_104,isw_topic_105,isw_topic_106,isw_topic_107,isw_topic_108,isw_topic_109,isw_topic_110,isw_topic_111,isw_topic_112,isw_topic_113,isw_topic_114,isw_topic_115,isw_topic_116,isw_topic_117,isw_topic_118,isw_topic_119,isw_topic_120,isw_topic_121,isw_topic_122,isw_topic_123,isw_topic_124,isw_topic_125,isw_topic_126,isw_topic_127,isw_topic_128,isw_topic_129,isw_topic_130,isw_topic_131,isw_topic_132,isw_topic_133,isw_topic_134,isw_topic_135,isw_topic_136,isw_topic_137,isw_topic_138,isw_topic_139,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149
0,2022-02-24 00:00:00,2,49.2336,28.4486,2.8,-0.3,80.5,0.3,100.0,4.17,0.0,0.0,27.7,10.8,312.0,1023.4,82.5,60.4,5.4,3.0,0.77,2.1,-0.5,91.76,0.0,0.0,0.0,16.6,8.6,279.6,1021.0,91.1,0.0,0.1,0.0,False,False,False,True,False,True,False,False,2022,2,3,0,Vinnytsia,Europe/Kiev,2022-02-24,06:58:49,17:40:52,00:00:00,Вінницька,0,0.0,0.0,0.0,0.0,0.0,NaN,0,1,NaN,NaN,1,0.648336,-0.079572,0.178041,0.061854,0.073453,-0.055809,-0.011311,0.047310,0.008354,0.006474,-0.047099,0.020496,0.050190,-0.038573,0.057228,-0.072471,0.045393,-0.116783,0.168249,0.167355,-0.025072,-0.073858,-0.097024,-0.146191,0.027983,0.082989,0.014423,0.058370,-0.033928,0.046265,0.006901,-0.049498,0.126117,-0.089039,-0.006767,-0.031686,-0.010346,0.025637,0.061278,0.162580,-0.028081,0.161291,-0.018217,-0.002657,0.038485,-0.000530,0.044413,0.107176,-0.050721,0.064279,-0.066677,0.016742,-0.050257,0.072549,-0.035201,-0.027279,0.017115,0.043631,0.024660,-0.025841,0.031320,-0.076959,-0.072986,-0.040465,0.003958,-0.022475,0.033208,-0.028056,-0.0054

In [18]:
df_final.isna().sum()

datetime_hour        0
region_id            0
city_latitude        0
city_longitude       0
day_temp             0
                  ... 
isw_topic_145     5760
isw_topic_146     5760
isw_topic_147     5760
isw_topic_148     5760
isw_topic_149     5760
Length: 216, dtype: int64

In [19]:
df_final.fillna(0, inplace=True)

,datetime_hour,region_id,city_latitude,city_longitude,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,day_snow,day_snowdepth,day_windgust,day_windspeed,day_winddir,day_pressure,day_cloudcover,day_solarradiation,day_solarenergy,day_uvindex,day_moonphase,hour_temp,hour_feelslike,hour_humidity,hour_precip,hour_precipprob,hour_snow,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,day_conditions_simple_Clear,day_conditions_simple_Cloudy,day_conditions_simple_Rain,day_conditions_simple_Snow,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,year,month,day_of_week,hour,city_name,city_timezone,day_datetime,day_sunrise,day_sunset,hour_datetime,region_key,alarm_active,alarm_minutes_in_hour,alarm_lag_1,alarm_lag_3,alarm_lag_6,alarm_lag_12,alarms_in_last_24h,is_weekend,is_night,total_active_alarms_lag1,neighbour_alarms,hours_since_last_alarm,isw_topic_0,isw_topic_1,isw_topic_2,isw_topic_3,isw_topic_4,isw_topic_5,isw_topic_6,isw_topic_7,isw_topic_8,isw_topic_9,isw_topic_10,isw_topic_11,isw_topic_12,isw_topic_13,isw_topic_14,isw_topic_15,isw_topic_16,isw_topic_17,isw_topic_18,isw_topic_19,isw_topic_20,isw_topic_21,isw_topic_22,isw_topic_23,isw_topic_24,isw_topic_25,isw_topic_26,isw_topic_27,isw_topic_28,isw_topic_29,isw_topic_30,isw_topic_31,isw_topic_32,isw_topic_33,isw_topic_34,isw_topic_35,isw_topic_36,isw_topic_37,isw_topic_38,isw_topic_39,isw_topic_40,isw_topic_41,isw_topic_42,isw_topic_43,isw_topic_44,isw_topic_45,isw_topic_46,isw_topic_47,isw_topic_48,isw_topic_49,isw_topic_50,isw_topic_51,isw_topic_52,isw_topic_53,isw_topic_54,isw_topic_55,isw_topic_56,isw_topic_57,isw_topic_58,isw_topic_59,isw_topic_60,isw_topic_61,isw_topic_62,isw_topic_63,isw_topic_64,isw_topic_65,isw_topic_66,isw_topic_67,isw_topic_68,isw_topic_69,isw_topic_70,isw_topic_71,isw_topic_72,isw_topic_73,isw_topic_74,isw_topic_75,isw_topic_76,isw_topic_77,isw_topic_78,isw_topic_79,isw_topic_80,isw_topic_81,isw_topic_82,isw_topic_83,isw_topic_84,isw_topic_85,isw_topic_86,isw_topic_87,isw_topic_88,isw_topic_89,isw_topic_90,isw_topic_91,isw_topic_92,isw_topic_93,isw_topic_94,isw_topic_95,isw_topic_96,isw_topic_97,isw_topic_98,isw_topic_99,isw_topic_100,isw_topic_101,isw_topic_102,isw_topic_103,isw_topic_104,isw_topic_105,isw_topic_106,isw_topic_107,isw_topic_108,isw_topic_109,isw_topic_110,isw_topic_111,isw_topic_112,isw_topic_113,isw_topic_114,isw_topic_115,isw_topic_116,isw_topic_117,isw_topic_118,isw_topic_119,isw_topic_120,isw_topic_121,isw_topic_122,isw_topic_123,isw_topic_124,isw_topic_125,isw_topic_126,isw_topic_127,isw_topic_128,isw_topic_129,isw_topic_130,isw_topic_131,isw_topic_132,isw_topic_133,isw_topic_134,isw_topic_135,isw_topic_136,isw_topic_137,isw_topic_138,isw_topic_139,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149
0,2022-02-24 00:00:00,2,49.2336,28.4486,2.8,-0.3,80.5,0.3,100.0,4.17,0.0,0.0,27.7,10.8,312.0,1023.4,82.5,60.4,5.4,3.0,0.77,2.1,-0.5,91.76,0.0,0.0,0.0,16.6,8.6,279.6,1021.0,91.1,0.0,0.1,0.0,False,False,False,True,False,True,False,False,2022,2,3,0,Vinnytsia,Europe/Kiev,2022-02-24,06:58:49,17:40:52,00:00:00,Вінницька,0,0.000000,0.0,0.0,0.0,0.0,0.0,0,1,0.0,0.0,1,0.648336,-0.079572,0.178041,0.061854,0.073453,-0.055809,-0.011311,0.047310,0.008354,0.006474,-0.047099,0.020496,0.050190,-0.038573,0.057228,-0.072471,0.045393,-0.116783,0.168249,0.167355,-0.025072,-0.073858,-0.097024,-0.146191,0.027983,0.082989,0.014423,0.058370,-0.033928,0.046265,0.006901,-0.049498,0.126117,-0.089039,-0.006767,-0.031686,-0.010346,0.025637,0.061278,0.162580,-0.028081,0.161291,-0.018217,-0.002657,0.038485,-0.000530,0.044413,0.107176,-0.050721,0.064279,-0.066677,0.016742,-0.050257,0.072549,-0.035201,-0.027279,0.017115,0.043631,0.024660,-0.025841,0.031320,-0.076959,-0.072986,-0.040465,0.003958,-0.022475,0.033208,-0.028056,-0

In [20]:
df_final.isna().sum()

datetime_hour     0
region_id         0
city_latitude     0
city_longitude    0
day_temp          0
                 ..
isw_topic_145     0
isw_topic_146     0
isw_topic_147     0
isw_topic_148     0
isw_topic_149     0
Length: 216, dtype: int64

### Feature engineering for isw topics 

In [21]:
topic_cols = [c for c in df_final.columns if "isw_topic_" in c]
df_isw_abs = df_final[topic_cols].abs()

df_final['isw_total_intensity'] = df_isw_abs.sum(axis=1)
df_final['isw_topic_std'] = df_isw_abs.std(axis=1)
df_final['isw_topic_max'] = df_isw_abs.max(axis=1)
df_final['isw_topic_mean'] = df_isw_abs.mean(axis=1)

df_final['isw_velocity_24h'] = (
    df_final.groupby('region_id')['isw_total_intensity']
    .diff(24)
    .fillna(0)
)

df_final['isw_intensity_ema'] = (df_final.groupby('region_id')['isw_total_intensity'].transform(lambda x: x.shift(1).ewm(span=24).mean()))

df_final['isw_topic_entropy'] = -(df_isw_abs.div(df_isw_abs.sum(axis=1), axis=0) * np.log(df_isw_abs.div(df_isw_abs.sum(axis=1), axis=0) + 1e-9)).sum(axis=1)

isw_cols = ['isw_velocity_24h', 'isw_intensity_ema', 'isw_topic_entropy']
df_final[isw_cols] = df_final[isw_cols].fillna(0)

C:\Users\Uw11\AppData\Local\Temp\ipykernel_22436\1269306822.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['isw_total_intensity'] = df_isw_abs.sum(axis=1)
C:\Users\Uw11\AppData\Local\Temp\ipykernel_22436\1269306822.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['isw_topic_std'] = df_isw_abs.std(axis=1)
C:\Users\Uw11\AppData\Local\Temp\ipykernel_22436\1269306822.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor per

In [22]:
df_final.sample(10)

,datetime_hour,region_id,city_latitude,city_longitude,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,day_snow,day_snowdepth,day_windgust,day_windspeed,day_winddir,day_pressure,day_cloudcover,day_solarradiation,day_solarenergy,day_uvindex,day_moonphase,hour_temp,hour_feelslike,hour_humidity,hour_precip,hour_precipprob,hour_snow,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,day_conditions_simple_Clear,day_conditions_simple_Cloudy,day_conditions_simple_Rain,day_conditions_simple_Snow,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,year,month,day_of_week,hour,city_name,city_timezone,day_datetime,day_sunrise,day_sunset,hour_datetime,region_key,alarm_active,alarm_minutes_in_hour,alarm_lag_1,alarm_lag_3,alarm_lag_6,alarm_lag_12,alarms_in_last_24h,is_weekend,is_night,total_active_alarms_lag1,neighbour_alarms,hours_since_last_alarm,isw_topic_0,isw_topic_1,isw_topic_2,isw_topic_3,isw_topic_4,isw_topic_5,isw_topic_6,isw_topic_7,isw_topic_8,isw_topic_9,isw_topic_10,isw_topic_11,isw_topic_12,isw_topic_13,isw_topic_14,isw_topic_15,isw_topic_16,isw_topic_17,isw_topic_18,isw_topic_19,isw_topic_20,isw_topic_21,isw_topic_22,isw_topic_23,isw_topic_24,isw_topic_25,isw_topic_26,isw_topic_27,isw_topic_28,isw_topic_29,isw_topic_30,isw_topic_31,isw_topic_32,isw_topic_33,isw_topic_34,isw_topic_35,isw_topic_36,isw_topic_37,isw_topic_38,isw_topic_39,isw_topic_40,isw_topic_41,isw_topic_42,isw_topic_43,isw_topic_44,isw_topic_45,isw_topic_46,isw_topic_47,isw_topic_48,isw_topic_49,isw_topic_50,isw_topic_51,isw_topic_52,isw_topic_53,isw_topic_54,isw_topic_55,isw_topic_56,isw_topic_57,isw_topic_58,isw_topic_59,isw_topic_60,isw_topic_61,isw_topic_62,isw_topic_63,isw_topic_64,isw_topic_65,isw_topic_66,isw_topic_67,isw_topic_68,isw_topic_69,isw_topic_70,isw_topic_71,isw_topic_72,isw_topic_73,isw_topic_74,isw_topic_75,isw_topic_76,isw_topic_77,isw_topic_78,isw_topic_79,isw_topic_80,isw_topic_81,isw_topic_82,isw_topic_83,isw_topic_84,isw_topic_85,isw_topic_86,isw_topic_87,isw_topic_88,isw_topic_89,isw_topic_90,isw_topic_91,isw_topic_92,isw_topic_93,isw_topic_94,isw_topic_95,isw_topic_96,isw_topic_97,isw_topic_98,isw_topic_99,isw_topic_100,isw_topic_101,isw_topic_102,isw_topic_103,isw_topic_104,isw_topic_105,isw_topic_106,isw_topic_107,isw_topic_108,isw_topic_109,isw_topic_110,isw_topic_111,isw_topic_112,isw_topic_113,isw_topic_114,isw_topic_115,isw_topic_116,isw_topic_117,isw_topic_118,isw_topic_119,isw_topic_120,isw_topic_121,isw_topic_122,isw_topic_123,isw_topic_124,isw_topic_125,isw_topic_126,isw_topic_127,isw_topic_128,isw_topic_129,isw_topic_130,isw_topic_131,isw_topic_132,isw_topic_133,isw_topic_134,isw_topic_135,isw_topic_136,isw_topic_137,isw_topic_138,isw_topic_139,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149,isw_total_intensity,isw_topic_std,isw_topic_max,isw_topic_mean,isw_velocity_24h,isw_intensity_ema,isw_topic_entropy
480528,2022-08-12 07:00:00,21,46.6401,32.6142,26.3,18.1,63.2,0.300,100.0,12.50,0.0,0.0,38.9,23.4,27.5,1010.9,64.0,142.4,12.2,5.0,0.48,22.4,22.4,67.11,0.000,0.0,0.0,34.9,23.4,8.4,1011.0,52.8,42.0,0.2,0.0,False,False,True,False,False,True,False,False,2022,8,4,7,Kherson,Europe/Kiev,2022-08-12,05:43:26,20:04:59,07:00:00,Херсонська,0,0.00,1.0,1.0,0.0,1.0,6.0,0,0,2.0,0.0,0,0.725403,-0.215571,0.199881,-0.060110,-0.080390,0.108017,0.010103,0.068705,0.138381,-0.108192,0.040550,-0.003107,-0.038401,-0.144589,-0.101464,0.073721,-0.108335,0.003920,-0.053107,-0.016416,-0.046681,0.041288,0.063766,-0.126378,0.114511,-0.055293,-0.113651,-0.095839,-0.006889,-0.045769,0.040402,0.002332,-0.009140,-0.036420,-0.021008,-0.031235,-0.005009,0.005115,-0.007556,-0.051869,-0.037470,-0.005588,0.032289,-0.025059,0.043514,-0.000166,-0.022387,0.002547,0.037568,-0.019907,0.008909,0.026504,-0.003893,0.012528,-0.

### TELEGRAM MERGE

In [23]:
df_tg_matrix = pd.read_csv("../data/telegram_processed_svd.csv")

In [24]:
df_tg_matrix.head()

,date,channel,tg_topic_0,tg_topic_1,tg_topic_2,tg_topic_3,tg_topic_4,tg_topic_5,tg_topic_6,tg_topic_7,tg_topic_8,tg_topic_9,tg_topic_10,tg_topic_11,tg_topic_12,tg_topic_13,tg_topic_14,tg_topic_15,tg_topic_16,tg_topic_17,tg_topic_18,tg_topic_19,tg_topic_20,tg_topic_21,tg_topic_22,tg_topic_23,tg_topic_24,tg_topic_25,tg_topic_26,tg_topic_27,tg_topic_28,tg_topic_29,tg_topic_30,tg_topic_31,tg_topic_32,tg_topic_33,tg_topic_34,tg_topic_35,tg_topic_36,tg_topic_37,tg_topic_38,tg_topic_39,tg_topic_40,tg_topic_41,tg_topic_42,tg_topic_43,tg_topic_44,tg_topic_45,tg_topic_46,tg_topic_47,tg_topic_48,tg_topic_49,tg_topic_50,tg_topic_51,tg_topic_52,tg_topic_53,tg_topic_54,tg_topic_55,tg_topic_56,tg_topic_57,tg_topic_58,tg_topic_59,tg_topic_60,tg_topic_61,tg_topic_62,tg_topic_63,tg_topic_64,tg_topic_65,tg_topic_66,tg_topic_67,tg_topic_68,tg_topic_69,tg_topic_70,tg_topic_71,tg_topic_72,tg_topic_73,tg_topic_74,tg_topic_75,tg_topic_76,tg_topic_77,tg_topic_78,tg_topic_79,tg_topic_80,tg_topic_81,tg_topic_82,tg_topic_83,tg_topic_84,tg_topic_85,tg_topic_86,tg_topic_87,tg_topic_88,tg_topic_89,tg_topic_90,tg_topic_91,tg_topic_92,tg_topic_93,tg_topic_94,tg_topic_95,tg_topic_96,tg_topic_97,tg_topic_98,tg_topic_99,tg_topic_100,tg_topic_101,tg_topic_102,tg_topic_103,tg_topic_104,tg_topic_105,tg_topic_106,tg_topic_107,tg_topic_108,tg_topic_109,tg_topic_110,tg_topic_111,tg_topic_112,tg_topic_113,tg_topic_114,tg_topic_115,tg_topic_116,tg_topic_117,tg_topic_118,tg_topic_119,tg_topic_120,tg_topic_121,tg_topic_122,tg_topic_123,tg_topic_124,tg_topic_125,tg_topic_126,tg_topic_127,tg_topic_128,tg_topic_129,tg_topic_130,tg_topic_131,tg_topic_132,tg_topic_133,tg_topic_134,tg_topic_135,tg_topic_136,tg_topic_137,tg_topic_138,tg_topic_139,tg_topic_140,tg_topic_141,tg_topic_142,tg_topic_143,tg_topic_144,tg_topic_145,tg_topic_146,tg_topic_147,tg_topic_148,tg_topic_149,tg_topic_150,tg_topic_151,tg_topic_152,tg_topic_153,tg_topic_154,tg_topic_155,tg_topic_156,tg_topic_157,tg_topic_158,tg_topic_159,tg_topic_160,tg_topic_161,tg_topic_162,tg_topic_163,tg_topic_164,tg_topic_165,tg_topic_166,tg_topic_167,tg_topic_168,tg_topic_169,tg_topic_170,tg_topic_171,tg_topic_172,tg_topic_173,tg_topic_174,tg_topic_175,tg_topic_176,tg_topic_177,tg_topic_178,tg_topic_179,tg_topic_180,tg_topic_181,tg_topic_182,tg_topic_183,tg_topic_184,tg_topic_185,tg_topic_186,tg_topic_187,tg_topic_188,tg_topic_189,tg_topic_190,tg_topic_191,tg_topic_192,tg_topic_193,tg_topic_194,tg_topic_195,tg_topic_196,tg_topic_197,tg_topic_198,tg_topic_199,tg_topic_200,tg_topic_201,tg_topic_202,tg_topic_203,tg_topic_204,tg_topic_205,tg_topic_206,tg_topic_207,tg_topic_208,tg_topic_209,tg_topic_210,tg_topic_211,tg_topic_212,tg_topic_213,tg_topic_214,tg_topic_215,tg_topic_216,tg_topic_217,tg_topic_218,tg_topic_219,tg_topic_220,tg_topic_221,tg_topic_222,tg_topic_223,tg_topic_224,tg_topic_225,tg_topic_226,tg_topic_227,tg_topic_228,tg_topic_229,tg_topic_230,tg_topic_231,tg_topic_232,tg_topic_233,tg_topic_234,tg_topic_235,tg_topic_236,tg_topic_237,tg_topic_238,tg_topic_239,tg_topic_240,tg_topic_241,tg_topic_242,tg_topic_243,tg_topic_244,tg_topic_245,tg_topic_246,tg_topic_247,tg_topic_248,tg_topic_249
0,2026-03-06 13:06:58,DeepStateUA,0.007121,0.000107,0.019843,0.000990,-0.011395,0.032670,-0.050983,0.061080,0.042624,-0.024341,-0.018764,0.023404,-0.017137,-0.358836,0.425288,0.017966,-0.107660,-0.032066,-0.045986,0.118212,0.091172,-0.248637,-0.001069,0.150763,0.030603,-0.074030,0.005803,0.005871,0.034871,0.017239,0.058098,0.029814,-0.010845,0.031486,-0.022705,0.005607,-0.006324,-0.010169,0.024500,-0.015581,-0.012708,0.005139,0.007857,0.016734,-0.040289,-0.019688,-0.026763,0.009381,-0.043500,-0.009525,-0.013075,-0.010473,-0.018150,0.005988,0.032611,0.000850,0.017872,0.020885,0.023899,-0.019290,-0.006676,-0.018478,-0.023263,-0.005497,-0.006460,0.026051,0.004991,0.019687,-0.022244,0.010929,0.007016,-0.067349,-0.035249,-0.005686,0.018747,0.039159,-0.053868,-0.078189,-0.074737,-0.082163,0.020498,0.084336,0.113470,-0.025871,-0.048314,0.096

In [25]:
df_tg_matrix["datetime_hour"] = pd.to_datetime(df_tg_matrix["date"]).dt.floor("h")

C:\Users\Uw11\AppData\Local\Temp\ipykernel_22436\2851151304.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_tg_matrix["datetime_hour"] = pd.to_datetime(df_tg_matrix["date"]).dt.floor("h")


In [26]:
topic_cols = [c for c in df_tg_matrix.columns if "tg_topic_" in c]

tg_hourly = (
    df_tg_matrix.groupby("datetime_hour")[topic_cols]
    .mean()
    .sort_index()
    .reset_index()
)

C:\Users\Uw11\AppData\Local\Temp\ipykernel_22436\2188712141.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  .reset_index()


In [27]:
df_tg_matrix.head()

,date,channel,tg_topic_0,tg_topic_1,tg_topic_2,tg_topic_3,tg_topic_4,tg_topic_5,tg_topic_6,tg_topic_7,tg_topic_8,tg_topic_9,tg_topic_10,tg_topic_11,tg_topic_12,tg_topic_13,tg_topic_14,tg_topic_15,tg_topic_16,tg_topic_17,tg_topic_18,tg_topic_19,tg_topic_20,tg_topic_21,tg_topic_22,tg_topic_23,tg_topic_24,tg_topic_25,tg_topic_26,tg_topic_27,tg_topic_28,tg_topic_29,tg_topic_30,tg_topic_31,tg_topic_32,tg_topic_33,tg_topic_34,tg_topic_35,tg_topic_36,tg_topic_37,tg_topic_38,tg_topic_39,tg_topic_40,tg_topic_41,tg_topic_42,tg_topic_43,tg_topic_44,tg_topic_45,tg_topic_46,tg_topic_47,tg_topic_48,tg_topic_49,tg_topic_50,tg_topic_51,tg_topic_52,tg_topic_53,tg_topic_54,tg_topic_55,tg_topic_56,tg_topic_57,tg_topic_58,tg_topic_59,tg_topic_60,tg_topic_61,tg_topic_62,tg_topic_63,tg_topic_64,tg_topic_65,tg_topic_66,tg_topic_67,tg_topic_68,tg_topic_69,tg_topic_70,tg_topic_71,tg_topic_72,tg_topic_73,tg_topic_74,tg_topic_75,tg_topic_76,tg_topic_77,tg_topic_78,tg_topic_79,tg_topic_80,tg_topic_81,tg_topic_82,tg_topic_83,tg_topic_84,tg_topic_85,tg_topic_86,tg_topic_87,tg_topic_88,tg_topic_89,tg_topic_90,tg_topic_91,tg_topic_92,tg_topic_93,tg_topic_94,tg_topic_95,tg_topic_96,tg_topic_97,tg_topic_98,tg_topic_99,tg_topic_100,tg_topic_101,tg_topic_102,tg_topic_103,tg_topic_104,tg_topic_105,tg_topic_106,tg_topic_107,tg_topic_108,tg_topic_109,tg_topic_110,tg_topic_111,tg_topic_112,tg_topic_113,tg_topic_114,tg_topic_115,tg_topic_116,tg_topic_117,tg_topic_118,tg_topic_119,tg_topic_120,tg_topic_121,tg_topic_122,tg_topic_123,tg_topic_124,tg_topic_125,tg_topic_126,tg_topic_127,tg_topic_128,tg_topic_129,tg_topic_130,tg_topic_131,tg_topic_132,tg_topic_133,tg_topic_134,tg_topic_135,tg_topic_136,tg_topic_137,tg_topic_138,tg_topic_139,tg_topic_140,tg_topic_141,tg_topic_142,tg_topic_143,tg_topic_144,tg_topic_145,tg_topic_146,tg_topic_147,tg_topic_148,tg_topic_149,tg_topic_150,tg_topic_151,tg_topic_152,tg_topic_153,tg_topic_154,tg_topic_155,tg_topic_156,tg_topic_157,tg_topic_158,tg_topic_159,tg_topic_160,tg_topic_161,tg_topic_162,tg_topic_163,tg_topic_164,tg_topic_165,tg_topic_166,tg_topic_167,tg_topic_168,tg_topic_169,tg_topic_170,tg_topic_171,tg_topic_172,tg_topic_173,tg_topic_174,tg_topic_175,tg_topic_176,tg_topic_177,tg_topic_178,tg_topic_179,tg_topic_180,tg_topic_181,tg_topic_182,tg_topic_183,tg_topic_184,tg_topic_185,tg_topic_186,tg_topic_187,tg_topic_188,tg_topic_189,tg_topic_190,tg_topic_191,tg_topic_192,tg_topic_193,tg_topic_194,tg_topic_195,tg_topic_196,tg_topic_197,tg_topic_198,tg_topic_199,tg_topic_200,tg_topic_201,tg_topic_202,tg_topic_203,tg_topic_204,tg_topic_205,tg_topic_206,tg_topic_207,tg_topic_208,tg_topic_209,tg_topic_210,tg_topic_211,tg_topic_212,tg_topic_213,tg_topic_214,tg_topic_215,tg_topic_216,tg_topic_217,tg_topic_218,tg_topic_219,tg_topic_220,tg_topic_221,tg_topic_222,tg_topic_223,tg_topic_224,tg_topic_225,tg_topic_226,tg_topic_227,tg_topic_228,tg_topic_229,tg_topic_230,tg_topic_231,tg_topic_232,tg_topic_233,tg_topic_234,tg_topic_235,tg_topic_236,tg_topic_237,tg_topic_238,tg_topic_239,tg_topic_240,tg_topic_241,tg_topic_242,tg_topic_243,tg_topic_244,tg_topic_245,tg_topic_246,tg_topic_247,tg_topic_248,tg_topic_249,datetime_hour
0,2026-03-06 13:06:58,DeepStateUA,0.007121,0.000107,0.019843,0.000990,-0.011395,0.032670,-0.050983,0.061080,0.042624,-0.024341,-0.018764,0.023404,-0.017137,-0.358836,0.425288,0.017966,-0.107660,-0.032066,-0.045986,0.118212,0.091172,-0.248637,-0.001069,0.150763,0.030603,-0.074030,0.005803,0.005871,0.034871,0.017239,0.058098,0.029814,-0.010845,0.031486,-0.022705,0.005607,-0.006324,-0.010169,0.024500,-0.015581,-0.012708,0.005139,0.007857,0.016734,-0.040289,-0.019688,-0.026763,0.009381,-0.043500,-0.009525,-0.013075,-0.010473,-0.018150,0.005988,0.032611,0.000850,0.017872,0.020885,0.023899,-0.019290,-0.006676,-0.018478,-0.023263,-0.005497,-0.006460,0.026051,0.004991,0.019687,-0.022244,0.010929,0.007016,-0.067349,-0.035249,-0.005686,0.018747,0.039159,-0.053868,-0.078189,-0.074737,-0.082163,0.020498,0.084336,0.113470,-0.025871,-

In [28]:
#tg_hourly["datetime_hour"] = tg_hourly["datetime_hour"] + pd.Timedelta(hours=1)        ТУТ ЗСУВ ТГ НА 1 ГОДИНУ 

In [29]:
df_final = df_final.merge(tg_hourly, on="datetime_hour", how="left")

In [30]:
df_final.sample(20)

,datetime_hour,region_id,city_latitude,city_longitude,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,day_snow,day_snowdepth,day_windgust,day_windspeed,day_winddir,day_pressure,day_cloudcover,day_solarradiation,day_solarenergy,day_uvindex,day_moonphase,hour_temp,hour_feelslike,hour_humidity,hour_precip,hour_precipprob,hour_snow,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,day_conditions_simple_Clear,day_conditions_simple_Cloudy,day_conditions_simple_Rain,day_conditions_simple_Snow,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,year,month,day_of_week,hour,city_name,city_timezone,day_datetime,day_sunrise,day_sunset,hour_datetime,region_key,alarm_active,alarm_minutes_in_hour,alarm_lag_1,alarm_lag_3,alarm_lag_6,alarm_lag_12,alarms_in_last_24h,is_weekend,is_night,total_active_alarms_lag1,neighbour_alarms,hours_since_last_alarm,isw_topic_0,isw_topic_1,isw_topic_2,isw_topic_3,isw_topic_4,isw_topic_5,isw_topic_6,isw_topic_7,isw_topic_8,isw_topic_9,isw_topic_10,isw_topic_11,isw_topic_12,isw_topic_13,isw_topic_14,isw_topic_15,isw_topic_16,isw_topic_17,isw_topic_18,isw_topic_19,isw_topic_20,isw_topic_21,isw_topic_22,isw_topic_23,isw_topic_24,isw_topic_25,isw_topic_26,isw_topic_27,isw_topic_28,isw_topic_29,isw_topic_30,isw_topic_31,isw_topic_32,isw_topic_33,isw_topic_34,isw_topic_35,isw_topic_36,isw_topic_37,isw_topic_38,isw_topic_39,isw_topic_40,isw_topic_41,isw_topic_42,isw_topic_43,isw_topic_44,isw_topic_45,isw_topic_46,isw_topic_47,isw_topic_48,isw_topic_49,isw_topic_50,isw_topic_51,isw_topic_52,isw_topic_53,isw_topic_54,isw_topic_55,isw_topic_56,isw_topic_57,isw_topic_58,isw_topic_59,isw_topic_60,isw_topic_61,isw_topic_62,isw_topic_63,isw_topic_64,isw_topic_65,isw_topic_66,isw_topic_67,isw_topic_68,isw_topic_69,isw_topic_70,isw_topic_71,isw_topic_72,isw_topic_73,isw_topic_74,isw_topic_75,isw_topic_76,isw_topic_77,isw_topic_78,isw_topic_79,isw_topic_80,isw_topic_81,isw_topic_82,isw_topic_83,isw_topic_84,isw_topic_85,isw_topic_86,isw_topic_87,isw_topic_88,isw_topic_89,isw_topic_90,isw_topic_91,isw_topic_92,isw_topic_93,isw_topic_94,isw_topic_95,isw_topic_96,isw_topic_97,isw_topic_98,isw_topic_99,isw_topic_100,isw_topic_101,isw_topic_102,isw_topic_103,isw_topic_104,isw_topic_105,isw_topic_106,isw_topic_107,isw_topic_108,isw_topic_109,isw_topic_110,isw_topic_111,isw_topic_112,isw_topic_113,isw_topic_114,isw_topic_115,isw_topic_116,isw_topic_117,isw_topic_118,isw_topic_119,isw_topic_120,isw_topic_121,isw_topic_122,isw_topic_123,isw_topic_124,isw_topic_125,isw_topic_126,isw_topic_127,isw_topic_128,isw_topic_129,isw_topic_130,isw_topic_131,isw_topic_132,isw_topic_133,isw_topic_134,isw_topic_135,isw_topic_136,isw_topic_137,isw_topic_138,isw_topic_139,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149,isw_total_intensity,isw_topic_std,isw_topic_max,isw_topic_mean,isw_velocity_24h,isw_intensity_ema,isw_topic_entropy,tg_topic_0,tg_topic_1,tg_topic_2,tg_topic_3,tg_topic_4,tg_topic_5,tg_topic_6,tg_topic_7,tg_topic_8,tg_topic_9,tg_topic_10,tg_topic_11,tg_topic_12,tg_topic_13,tg_topic_14,tg_topic_15,tg_topic_16,tg_topic_17,tg_topic_18,tg_topic_19,tg_topic_20,tg_topic_21,tg_topic_22,tg_topic_23,tg_topic_24,tg_topic_25,tg_topic_26,tg_topic_27,tg_topic_28,tg_topic_29,tg_topic_30,tg_topic_31,tg_topic_32,tg_topic_33,tg_topic_34,tg_topic_35,tg_topic_36,tg_topic_37,tg_topic_38,tg_topic_39,tg_topic_40,tg_topic_41,tg_topic_42,tg_topic_43,tg_topic_44,tg_topic_45,tg_topic_46,tg_topic_47,tg_topic_48,tg_topic_49,tg_topic_50,tg_topic_51,tg_topic_52,tg_topic_53,tg_topic_54,tg_topic_55,tg_topic_56,tg_topic_57,tg_topic_58,tg_topic_59,tg_topic_60,tg_topic_61,tg_topic_62,tg_topic_63,tg_topic_64,tg_topic_65,tg_topic_66,tg_topic_67,tg_topic_68,tg_topic_69,tg_topic_70,tg_topic_71,tg_topic_72,tg_topic_73,tg_topic_74,t

In [31]:
df_final.shape

(635256, 473)

In [32]:
df_final = df_final.sort_values(["datetime_hour"])

In [33]:
with pd.option_context('display.max_rows', None):
    print(df_final.isnull().sum())

datetime_hour                         0
region_id                             0
city_latitude                         0
city_longitude                        0
day_temp                              0
day_dew                               0
day_humidity                          0
day_precip                            0
day_precipprob                        0
day_precipcover                       0
day_snow                              0
day_snowdepth                         0
day_windgust                          0
day_windspeed                         0
day_winddir                           0
day_pressure                          0
day_cloudcover                        0
day_solarradiation                    0
day_solarenergy                       0
day_uvindex                           0
day_moonphase                         0
hour_temp                             0
hour_feelslike                        0
hour_humidity                         0
hour_precip                           0


In [34]:
df_final = df_final.fillna(0)

In [35]:
tg_cols = [c for c in df_final.columns if 'tg_topic_' in c]
df_tg_abs = df_final[tg_cols].abs()

df_final['tg_total_intensity'] = df_tg_abs.sum(axis=1)
df_final['tg_topic_std'] = df_tg_abs.std(axis=1)
df_final['tg_topic_max'] = df_tg_abs.max(axis=1)
df_final['tg_topic_entropy'] = -(df_tg_abs.div(df_tg_abs.sum(axis=1), axis=0) * np.log(df_tg_abs.div(df_tg_abs.sum(axis=1), axis=0) + 1e-9)).sum(axis=1)

C:\Users\Uw11\AppData\Local\Temp\ipykernel_22436\409095542.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['tg_total_intensity'] = df_tg_abs.sum(axis=1)
C:\Users\Uw11\AppData\Local\Temp\ipykernel_22436\409095542.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['tg_topic_std'] = df_tg_abs.std(axis=1)
C:\Users\Uw11\AppData\Local\Temp\ipykernel_22436\409095542.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performanc

In [36]:
df_final['tg_velocity_3h'] = (df_final.groupby('region_id')['tg_total_intensity'].diff(3).fillna(0))
df_final['tg_intensity_ema_6h'] = (df_final.groupby('region_id')['tg_total_intensity'].transform(lambda x: x.ewm(span=6).mean()))
df_final['tg_intensity_zscore'] = (df_final.groupby('region_id')['tg_total_intensity'].transform(lambda x: (x - x.rolling(24, min_periods=1).mean()) / (x.rolling(24, min_periods=1).std() + 1e-9)))

C:\Users\Uw11\AppData\Local\Temp\ipykernel_22436\3857173777.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['tg_velocity_3h'] = (df_final.groupby('region_id')['tg_total_intensity'].diff(3).fillna(0))
C:\Users\Uw11\AppData\Local\Temp\ipykernel_22436\3857173777.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['tg_intensity_ema_6h'] = (df_final.groupby('region_id')['tg_total_intensity'].transform(lambda x: x.ewm(span=6).mean()))
C:\Users\Uw11\AppData\Local\Temp\ipykernel_22436\3857173777.py:3: PerformanceWa

In [37]:
df_final.sample(20)

,datetime_hour,region_id,city_latitude,city_longitude,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,day_snow,day_snowdepth,day_windgust,day_windspeed,day_winddir,day_pressure,day_cloudcover,day_solarradiation,day_solarenergy,day_uvindex,day_moonphase,hour_temp,hour_feelslike,hour_humidity,hour_precip,hour_precipprob,hour_snow,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,day_conditions_simple_Clear,day_conditions_simple_Cloudy,day_conditions_simple_Rain,day_conditions_simple_Snow,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,year,month,day_of_week,hour,city_name,city_timezone,day_datetime,day_sunrise,day_sunset,hour_datetime,region_key,alarm_active,alarm_minutes_in_hour,alarm_lag_1,alarm_lag_3,alarm_lag_6,alarm_lag_12,alarms_in_last_24h,is_weekend,is_night,total_active_alarms_lag1,neighbour_alarms,hours_since_last_alarm,isw_topic_0,isw_topic_1,isw_topic_2,isw_topic_3,isw_topic_4,isw_topic_5,isw_topic_6,isw_topic_7,isw_topic_8,isw_topic_9,isw_topic_10,isw_topic_11,isw_topic_12,isw_topic_13,isw_topic_14,isw_topic_15,isw_topic_16,isw_topic_17,isw_topic_18,isw_topic_19,isw_topic_20,isw_topic_21,isw_topic_22,isw_topic_23,isw_topic_24,isw_topic_25,isw_topic_26,isw_topic_27,isw_topic_28,isw_topic_29,isw_topic_30,isw_topic_31,isw_topic_32,isw_topic_33,isw_topic_34,isw_topic_35,isw_topic_36,isw_topic_37,isw_topic_38,isw_topic_39,isw_topic_40,isw_topic_41,isw_topic_42,isw_topic_43,isw_topic_44,isw_topic_45,isw_topic_46,isw_topic_47,isw_topic_48,isw_topic_49,isw_topic_50,isw_topic_51,isw_topic_52,isw_topic_53,isw_topic_54,isw_topic_55,isw_topic_56,isw_topic_57,isw_topic_58,isw_topic_59,isw_topic_60,isw_topic_61,isw_topic_62,isw_topic_63,isw_topic_64,isw_topic_65,isw_topic_66,isw_topic_67,isw_topic_68,isw_topic_69,isw_topic_70,isw_topic_71,isw_topic_72,isw_topic_73,isw_topic_74,isw_topic_75,isw_topic_76,isw_topic_77,isw_topic_78,isw_topic_79,isw_topic_80,isw_topic_81,isw_topic_82,isw_topic_83,isw_topic_84,isw_topic_85,isw_topic_86,isw_topic_87,isw_topic_88,isw_topic_89,isw_topic_90,isw_topic_91,isw_topic_92,isw_topic_93,isw_topic_94,isw_topic_95,isw_topic_96,isw_topic_97,isw_topic_98,isw_topic_99,isw_topic_100,isw_topic_101,isw_topic_102,isw_topic_103,isw_topic_104,isw_topic_105,isw_topic_106,isw_topic_107,isw_topic_108,isw_topic_109,isw_topic_110,isw_topic_111,isw_topic_112,isw_topic_113,isw_topic_114,isw_topic_115,isw_topic_116,isw_topic_117,isw_topic_118,isw_topic_119,isw_topic_120,isw_topic_121,isw_topic_122,isw_topic_123,isw_topic_124,isw_topic_125,isw_topic_126,isw_topic_127,isw_topic_128,isw_topic_129,isw_topic_130,isw_topic_131,isw_topic_132,isw_topic_133,isw_topic_134,isw_topic_135,isw_topic_136,isw_topic_137,isw_topic_138,isw_topic_139,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149,isw_total_intensity,isw_topic_std,isw_topic_max,isw_topic_mean,isw_velocity_24h,isw_intensity_ema,isw_topic_entropy,tg_topic_0,tg_topic_1,tg_topic_2,tg_topic_3,tg_topic_4,tg_topic_5,tg_topic_6,tg_topic_7,tg_topic_8,tg_topic_9,tg_topic_10,tg_topic_11,tg_topic_12,tg_topic_13,tg_topic_14,tg_topic_15,tg_topic_16,tg_topic_17,tg_topic_18,tg_topic_19,tg_topic_20,tg_topic_21,tg_topic_22,tg_topic_23,tg_topic_24,tg_topic_25,tg_topic_26,tg_topic_27,tg_topic_28,tg_topic_29,tg_topic_30,tg_topic_31,tg_topic_32,tg_topic_33,tg_topic_34,tg_topic_35,tg_topic_36,tg_topic_37,tg_topic_38,tg_topic_39,tg_topic_40,tg_topic_41,tg_topic_42,tg_topic_43,tg_topic_44,tg_topic_45,tg_topic_46,tg_topic_47,tg_topic_48,tg_topic_49,tg_topic_50,tg_topic_51,tg_topic_52,tg_topic_53,tg_topic_54,tg_topic_55,tg_topic_56,tg_topic_57,tg_topic_58,tg_topic_59,tg_topic_60,tg_topic_61,tg_topic_62,tg_topic_63,tg_topic_64,tg_topic_65,tg_topic_66,tg_topic_67,tg_topic_68,tg_topic_69,tg_topic_70,tg_topic_71,tg_topic_72,tg_topic_73,tg_topic_74,t

In [38]:
df_final.to_parquet("merged_dataset.parquet", index=False, engine="pyarrow")

# ТУТ БУДЕ EDA

## Weather, ISW, Telegram shift for further models trainings

In [39]:
df_to_train = df_final.copy()
df_to_train = df_to_train.sort_values(['region_id', 'datetime_hour'])

In [40]:
# delete unnessecary columns
columns_to_delete = ['city_latitude', 'city_longitude', 'day_temp', 'day_dew', 'day_humidity', 'day_precip', 'day_precipprob', 'day_precipcover', 
                     'day_snow', 'day_snowdepth', 'day_windgust', 'day_windspeed', 'day_winddir', 'day_pressure', 'day_cloudcover', 'day_solarradiation',
                     'day_solarenergy', 'day_uvindex', 'day_moonphase', 'day_conditions_simple_Clear', 'day_conditions_simple_Cloudy', 'day_conditions_simple_Rain',
                     'day_conditions_simple_Snow', 'year', 'month', 'hour', 'day_of_week', 'city_timezone', 'day_datetime', 'hour_datetime', 'region_key']
df_to_train = df_to_train.drop(columns=columns_to_delete)

In [41]:
isw_cols = [c for c in df_to_train.columns if 'isw_' in c]
df_to_train[isw_cols] = df_to_train.groupby('region_id')[isw_cols].shift(24).fillna(0)

In [42]:
df_to_train.head()

,datetime_hour,region_id,hour_temp,hour_feelslike,hour_humidity,hour_precip,hour_precipprob,hour_snow,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,city_name,day_sunrise,day_sunset,alarm_active,alarm_minutes_in_hour,alarm_lag_1,alarm_lag_3,alarm_lag_6,alarm_lag_12,alarms_in_last_24h,is_weekend,is_night,total_active_alarms_lag1,neighbour_alarms,hours_since_last_alarm,isw_topic_0,isw_topic_1,isw_topic_2,isw_topic_3,isw_topic_4,isw_topic_5,isw_topic_6,isw_topic_7,isw_topic_8,isw_topic_9,isw_topic_10,isw_topic_11,isw_topic_12,isw_topic_13,isw_topic_14,isw_topic_15,isw_topic_16,isw_topic_17,isw_topic_18,isw_topic_19,isw_topic_20,isw_topic_21,isw_topic_22,isw_topic_23,isw_topic_24,isw_topic_25,isw_topic_26,isw_topic_27,isw_topic_28,isw_topic_29,isw_topic_30,isw_topic_31,isw_topic_32,isw_topic_33,isw_topic_34,isw_topic_35,isw_topic_36,isw_topic_37,isw_topic_38,isw_topic_39,isw_topic_40,isw_topic_41,isw_topic_42,isw_topic_43,isw_topic_44,isw_topic_45,isw_topic_46,isw_topic_47,isw_topic_48,isw_topic_49,isw_topic_50,isw_topic_51,isw_topic_52,isw_topic_53,isw_topic_54,isw_topic_55,isw_topic_56,isw_topic_57,isw_topic_58,isw_topic_59,isw_topic_60,isw_topic_61,isw_topic_62,isw_topic_63,isw_topic_64,isw_topic_65,isw_topic_66,isw_topic_67,isw_topic_68,isw_topic_69,isw_topic_70,isw_topic_71,isw_topic_72,isw_topic_73,isw_topic_74,isw_topic_75,isw_topic_76,isw_topic_77,isw_topic_78,isw_topic_79,isw_topic_80,isw_topic_81,isw_topic_82,isw_topic_83,isw_topic_84,isw_topic_85,isw_topic_86,isw_topic_87,isw_topic_88,isw_topic_89,isw_topic_90,isw_topic_91,isw_topic_92,isw_topic_93,isw_topic_94,isw_topic_95,isw_topic_96,isw_topic_97,isw_topic_98,isw_topic_99,isw_topic_100,isw_topic_101,isw_topic_102,isw_topic_103,isw_topic_104,isw_topic_105,isw_topic_106,isw_topic_107,isw_topic_108,isw_topic_109,isw_topic_110,isw_topic_111,isw_topic_112,isw_topic_113,isw_topic_114,isw_topic_115,isw_topic_116,isw_topic_117,isw_topic_118,isw_topic_119,isw_topic_120,isw_topic_121,isw_topic_122,isw_topic_123,isw_topic_124,isw_topic_125,isw_topic_126,isw_topic_127,isw_topic_128,isw_topic_129,isw_topic_130,isw_topic_131,isw_topic_132,isw_topic_133,isw_topic_134,isw_topic_135,isw_topic_136,isw_topic_137,isw_topic_138,isw_topic_139,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149,isw_total_intensity,isw_topic_std,isw_topic_max,isw_topic_mean,isw_velocity_24h,isw_intensity_ema,isw_topic_entropy,tg_topic_0,tg_topic_1,tg_topic_2,tg_topic_3,tg_topic_4,tg_topic_5,tg_topic_6,tg_topic_7,tg_topic_8,tg_topic_9,tg_topic_10,tg_topic_11,tg_topic_12,tg_topic_13,tg_topic_14,tg_topic_15,tg_topic_16,tg_topic_17,tg_topic_18,tg_topic_19,tg_topic_20,tg_topic_21,tg_topic_22,tg_topic_23,tg_topic_24,tg_topic_25,tg_topic_26,tg_topic_27,tg_topic_28,tg_topic_29,tg_topic_30,tg_topic_31,tg_topic_32,tg_topic_33,tg_topic_34,tg_topic_35,tg_topic_36,tg_topic_37,tg_topic_38,tg_topic_39,tg_topic_40,tg_topic_41,tg_topic_42,tg_topic_43,tg_topic_44,tg_topic_45,tg_topic_46,tg_topic_47,tg_topic_48,tg_topic_49,tg_topic_50,tg_topic_51,tg_topic_52,tg_topic_53,tg_topic_54,tg_topic_55,tg_topic_56,tg_topic_57,tg_topic_58,tg_topic_59,tg_topic_60,tg_topic_61,tg_topic_62,tg_topic_63,tg_topic_64,tg_topic_65,tg_topic_66,tg_topic_67,tg_topic_68,tg_topic_69,tg_topic_70,tg_topic_71,tg_topic_72,tg_topic_73,tg_topic_74,tg_topic_75,tg_topic_76,tg_topic_77,tg_topic_78,tg_topic_79,tg_topic_80,tg_topic_81,tg_topic_82,tg_topic_83,tg_topic_84,tg_topic_85,tg_topic_86,tg_topic_87,tg_topic_88,tg_topic_89,tg_topic_90,tg_topic_91,tg_topic_92,tg_topic_93,tg_topic_94,tg_topic_95,tg_topic_96,tg_topic_97,tg_topic_98,tg_topic_99,tg_topic_100,tg_topic_101,tg_topic_102,tg_topic_103,tg_topic_104,tg_topic_105,tg_topic_106,tg_topic_107,tg_topic_108,tg_topic_109,tg_topic_110,t

In [43]:
tg_cols = [c for c in df_to_train.columns if 'tg_' in c]
df_to_train[tg_cols] = df_to_train.groupby('region_id')[tg_cols].shift(1).fillna(0)

In [44]:
hour_weather_cols = [c for c in df_to_train.columns if c.startswith('hour_')]
df_to_train[hour_weather_cols] = (df_to_train.groupby('region_id')[hour_weather_cols].shift(1).fillna(0))

In [46]:
df_to_train.isna().sum()

datetime_hour          0
region_id              0
hour_temp              0
hour_feelslike         0
hour_humidity          0
                      ..
tg_topic_max           0
tg_topic_entropy       0
tg_velocity_3h         0
tg_intensity_ema_6h    0
tg_intensity_zscore    0
Length: 449, dtype: int64

In [47]:
df_to_train['alarm_minutes_in_hour'] = (df_to_train.groupby('region_id')['alarm_minutes_in_hour'].shift(1).fillna(0))

In [48]:
df_to_train.head()

,datetime_hour,region_id,hour_temp,hour_feelslike,hour_humidity,hour_precip,hour_precipprob,hour_snow,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,city_name,day_sunrise,day_sunset,alarm_active,alarm_minutes_in_hour,alarm_lag_1,alarm_lag_3,alarm_lag_6,alarm_lag_12,alarms_in_last_24h,is_weekend,is_night,total_active_alarms_lag1,neighbour_alarms,hours_since_last_alarm,isw_topic_0,isw_topic_1,isw_topic_2,isw_topic_3,isw_topic_4,isw_topic_5,isw_topic_6,isw_topic_7,isw_topic_8,isw_topic_9,isw_topic_10,isw_topic_11,isw_topic_12,isw_topic_13,isw_topic_14,isw_topic_15,isw_topic_16,isw_topic_17,isw_topic_18,isw_topic_19,isw_topic_20,isw_topic_21,isw_topic_22,isw_topic_23,isw_topic_24,isw_topic_25,isw_topic_26,isw_topic_27,isw_topic_28,isw_topic_29,isw_topic_30,isw_topic_31,isw_topic_32,isw_topic_33,isw_topic_34,isw_topic_35,isw_topic_36,isw_topic_37,isw_topic_38,isw_topic_39,isw_topic_40,isw_topic_41,isw_topic_42,isw_topic_43,isw_topic_44,isw_topic_45,isw_topic_46,isw_topic_47,isw_topic_48,isw_topic_49,isw_topic_50,isw_topic_51,isw_topic_52,isw_topic_53,isw_topic_54,isw_topic_55,isw_topic_56,isw_topic_57,isw_topic_58,isw_topic_59,isw_topic_60,isw_topic_61,isw_topic_62,isw_topic_63,isw_topic_64,isw_topic_65,isw_topic_66,isw_topic_67,isw_topic_68,isw_topic_69,isw_topic_70,isw_topic_71,isw_topic_72,isw_topic_73,isw_topic_74,isw_topic_75,isw_topic_76,isw_topic_77,isw_topic_78,isw_topic_79,isw_topic_80,isw_topic_81,isw_topic_82,isw_topic_83,isw_topic_84,isw_topic_85,isw_topic_86,isw_topic_87,isw_topic_88,isw_topic_89,isw_topic_90,isw_topic_91,isw_topic_92,isw_topic_93,isw_topic_94,isw_topic_95,isw_topic_96,isw_topic_97,isw_topic_98,isw_topic_99,isw_topic_100,isw_topic_101,isw_topic_102,isw_topic_103,isw_topic_104,isw_topic_105,isw_topic_106,isw_topic_107,isw_topic_108,isw_topic_109,isw_topic_110,isw_topic_111,isw_topic_112,isw_topic_113,isw_topic_114,isw_topic_115,isw_topic_116,isw_topic_117,isw_topic_118,isw_topic_119,isw_topic_120,isw_topic_121,isw_topic_122,isw_topic_123,isw_topic_124,isw_topic_125,isw_topic_126,isw_topic_127,isw_topic_128,isw_topic_129,isw_topic_130,isw_topic_131,isw_topic_132,isw_topic_133,isw_topic_134,isw_topic_135,isw_topic_136,isw_topic_137,isw_topic_138,isw_topic_139,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149,isw_total_intensity,isw_topic_std,isw_topic_max,isw_topic_mean,isw_velocity_24h,isw_intensity_ema,isw_topic_entropy,tg_topic_0,tg_topic_1,tg_topic_2,tg_topic_3,tg_topic_4,tg_topic_5,tg_topic_6,tg_topic_7,tg_topic_8,tg_topic_9,tg_topic_10,tg_topic_11,tg_topic_12,tg_topic_13,tg_topic_14,tg_topic_15,tg_topic_16,tg_topic_17,tg_topic_18,tg_topic_19,tg_topic_20,tg_topic_21,tg_topic_22,tg_topic_23,tg_topic_24,tg_topic_25,tg_topic_26,tg_topic_27,tg_topic_28,tg_topic_29,tg_topic_30,tg_topic_31,tg_topic_32,tg_topic_33,tg_topic_34,tg_topic_35,tg_topic_36,tg_topic_37,tg_topic_38,tg_topic_39,tg_topic_40,tg_topic_41,tg_topic_42,tg_topic_43,tg_topic_44,tg_topic_45,tg_topic_46,tg_topic_47,tg_topic_48,tg_topic_49,tg_topic_50,tg_topic_51,tg_topic_52,tg_topic_53,tg_topic_54,tg_topic_55,tg_topic_56,tg_topic_57,tg_topic_58,tg_topic_59,tg_topic_60,tg_topic_61,tg_topic_62,tg_topic_63,tg_topic_64,tg_topic_65,tg_topic_66,tg_topic_67,tg_topic_68,tg_topic_69,tg_topic_70,tg_topic_71,tg_topic_72,tg_topic_73,tg_topic_74,tg_topic_75,tg_topic_76,tg_topic_77,tg_topic_78,tg_topic_79,tg_topic_80,tg_topic_81,tg_topic_82,tg_topic_83,tg_topic_84,tg_topic_85,tg_topic_86,tg_topic_87,tg_topic_88,tg_topic_89,tg_topic_90,tg_topic_91,tg_topic_92,tg_topic_93,tg_topic_94,tg_topic_95,tg_topic_96,tg_topic_97,tg_topic_98,tg_topic_99,tg_topic_100,tg_topic_101,tg_topic_102,tg_topic_103,tg_topic_104,tg_topic_105,tg_topic_106,tg_topic_107,tg_topic_108,tg_topic_109,tg_topic_110,t